# 7차시 실습 — 설비·공정 비교와 수율·불량 분석

강의에서 배운 순서대로 위에서 아래로 실행합니다(셀을 클릭하고 **Shift + Enter**).
`____`(밑줄 네 개)로 비어 있는 곳만 채우면 됩니다.

- 오늘 데이터는 교육용 **가상 데이터**입니다(300행 = 로트 75개 × 웨이퍼 4장).
- `합격여부` 열: **1 = 합격, -1 = 불합격**.
- 위 셀을 건너뛰면 `NameError`가 납니다. 이상하면 "커널 → 다시 시작 후 모두 실행".

## 1단계. 데이터 읽고 전체 합격률 보기
CSV를 읽어 행·열 개수를 확인합니다.

**주의**: 경로 앞의 `../../`까지 그대로 써야 합니다. 경로가 틀리면 `FileNotFoundError`가 납니다.

In [ ]:
import pandas as pd

# TODO: CSV 파일을 읽는 pandas 명령을 채우세요.
df = pd.____("../../data/weekly/week07/week07_equipment_yield.csv", encoding="utf-8-sig")
print(df.shape)

In [ ]:
df.head()  # (중간 확인)

### 합격 열 만들고 전체 합격률 구하기
`합격여부`가 1이면 True(합격), 아니면 False인 `합격` 열을 만듭니다.
True는 1, False는 0으로 계산되므로 `sum()`은 합격 장수, `mean()`은 합격 비율입니다.

- 첫 번째 빈칸: 합격을 뜻하는 숫자
- 두 번째 빈칸: 평균을 구하는 함수 이름

In [ ]:
# TODO: 빈칸 두 곳을 채우세요.
df["합격"] = df["합격여부"] == ____
print(df["합격"].sum(), "장 합격")
print(df["합격"].____() * 100)

## 2단계. groupby 감 잡기 — 반별 평균 점수
학생 6명을 반별로 묶어 평균 점수를 구합니다. "같은 이름표(반)끼리 모아서 → 바구니마다 평균".

In [ ]:
toy = pd.DataFrame({
    "이름": ["지민", "하은", "서준", "예린", "도윤", "시우"],
    "반": ["A", "A", "B", "B", "C", "C"],
    "점수": [90, 80, 70, 60, 85, 95],
})

# TODO: 같은 반끼리 묶는 명령을 채우세요.
print(toy.____("반")["점수"].mean())

## 3단계. 설비별로 묶어 비교하기
먼저 설비마다 웨이퍼를 몇 장 처리했는지(`count`), 그중 몇 장이 합격했는지(`sum`) 봅니다.

**주의**: 설비마다 처리한 장수가 다릅니다. 그래서 합격 "장수"가 아니라 "비율"로 비교합니다.

In [ ]:
# TODO: 바구니마다 개수를 세는 함수 이름을 채우세요.
print(df.groupby("설비번호")["합격"].____())

In [ ]:
print(df.groupby("설비번호")["합격"].sum())

### 설비별 합격률
`pass_rate`는 뒤 단계(8·10단계)에서도 씁니다. `.round(1)`은 소수 첫째 자리까지 반올림해서 보여 줍니다.

In [ ]:
# TODO: 무엇을 기준으로 묶을지 열 이름을 채우세요.
pass_rate = df.groupby(____)["합격"].mean() * 100
print(pass_rate.round(1))

### 설비별 평균 온도
합격률이 크게 달랐던 설비들이 평균 온도도 크게 다른지 확인합니다.

In [ ]:
# TODO: 평균을 낼 열 이름(온도)을 채우세요.
print(df.groupby("설비번호")[____].mean().round(1))

## 4단계. 로트별 수율 계산하기
이번에는 로트번호로 묶어 로트마다 수율(%)을 구합니다. 로트 하나에 웨이퍼가 4장이라
수율은 0·25·50·75·100% 중 하나입니다.

In [ ]:
# TODO: 로트마다 묶을 수 있게 열 이름을 채우세요.
lot_yield = df.groupby(____)["합격"].mean() * 100
print(lot_yield.head())

In [ ]:
# TODO: 수율이 80보다 "작은" 로트를 세도록 비교 기호를 채우세요.
print((lot_yield ____ 80).sum(), "개 로트가 수율 80% 미만")

### 합격률이 가장 낮았던 EQ-04의 로트만 보기
EQ-04 기록만 골라낸 뒤 같은 방법으로 로트별 수율을 구합니다.

In [ ]:
# TODO: 골라낼 설비 이름을 채우세요(글자는 따옴표로).
eq04 = df[df["설비번호"] == ____]
eq04_lot_yield = eq04.groupby("로트번호")["합격"].mean() * 100
print(len(eq04_lot_yield), "개 로트")
print((eq04_lot_yield < 80).sum(), "개 로트가 수율 80% 미만")
print(eq04_lot_yield.value_counts())

### 수율이 가장 낮은 로트 열어 보기
수율이 가장 낮은 로트를 찾고, 그 로트의 기록을 여러 열과 함께 봅니다.

In [ ]:
print(lot_yield.sort_values().head(1))

- 빈칸: 바로 위 결과에 나온 로트번호(따옴표로 감싸기)

In [ ]:
# TODO: 위에서 찾은 로트번호를 채우세요.
print(df[df["로트번호"] == ____][["설비번호", "공정명", "작업조", "불량유형"]])

## 5단계. 불량 유형 줄 세우기(파레토)
불합격 웨이퍼만 골라 불량 유형별 건수를 셉니다. `value_counts()`는 많은 순서로 줄을 세워 줍니다.

**주의**: 불합격은 `-1`입니다(1이 합격).

In [ ]:
# TODO: 불합격을 뜻하는 숫자를 채우세요.
defect_counts = df[df["합격여부"] == ____]["불량유형"].value_counts()
print(defect_counts)

### 막대그래프로 그리기
6차시에 배운 `plt.bar(이름들, 숫자들)` 틀 그대로입니다. 높이는 `defect_counts`(건수)입니다.

- 빈칸: 막대 아래 이름표(불량 유형 이름)를 꺼내는 부분. 6차시 `plt.bar(counts.index, counts)`의 첫 번째 자리와 같은 모양
- `plt.xticks(rotation=45)`는 이름표 글자를 45도 기울여 겹치지 않게 합니다(6차시 선 그래프에서 쓴 줄).

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# TODO: 막대 아래 이름표를 채우세요.
plt.bar(defect_counts.____, defect_counts)
plt.xticks(rotation=45)
plt.xlabel("불량유형")
plt.ylabel("불량 건수")
plt.show()

## 6단계. 결과를 문장으로 정리하기
3·4단계 결과를 보고 아래 문장의 빈칸을 채우세요(다음 셀에 적기).

> 합격률이 가장 낮은 설비는 ____(____%)이고, 수율 80% 미만 로트 ____개 중 ____개가 이 설비에서 나왔다.
> 하지만 이 설비의 로트 22개 중 ____개는 수율 100%였다. 그래서 이 설비는 원인이라고 단정하지 않고 "____"(으)로 더 살펴본다.

**주의**: 눈에 띈다고 바로 "원인"이라고 쓰지 않습니다(1차시 "용의자 ≠ 범인").

**내 문장**: (여기에 적어보세요)

### 도전 문제 — 설비×작업조 두 기준으로 묶기 (빠른 학습자용)
설비번호와 작업조 **두 열**로 함께 묶어 조합마다 합격률과 장수를 봅니다.

- 첫 번째 빈칸: 묶을 열 이름 두 개를 대괄호 `[ ]` 하나로 감싼 것(강의 "도전 실습"의 코드)
- 두 번째 빈칸: 3단계에서 장수를 센 함수 이름

**주의**: 장수가 적은 조합의 합격률은 1장만 달라져도 크게 흔들립니다. 합격률 옆에 장수를 꼭 함께 보세요.

In [ ]:
# TODO: 두 열로 함께 묶도록 빈칸을 채우세요.
team_rate = df.groupby(____)["합격"].mean() * 100
print(team_rate.round(1))

In [ ]:
# TODO: 조합마다 장수를 세는 함수 이름을 채우세요.
print(df.groupby(["설비번호", "작업조"])["합격"].____())

## 7단계. AI에게 질문하며 더 알아보기
아래 질문 중 하나 이상을 AI 챗봇에게 물어보고, 새로 알게 된 점을 다음 셀에 한두 줄로 적으세요.

- "pandas의 groupby를 택배 분류 센터에 비유해서 3줄로 설명해주세요."
- "수율과 불량률은 어떻게 계산하나요? 사과 따기 예시로 알려주세요."
- "파레토 차트는 왜 많은 것부터 줄을 세우나요?"

**주의**: AI의 답도 틀릴 수 있습니다. 강의 내용과 비교해 보세요.

**내가 새로 알게 된 점**: (여기에 적어보세요)

## 8단계. AI에게 코드 생성 요청하고 직접 실행해보기
AI에게 아래처럼 요청하고, 받은 코드를 다음 셀에 붙여넣어 실행하세요.

> pass_rate라는 pandas Series에 설비별 합격률(%)이 들어 있어. 90% 미만인 설비는 '점검 필요', 아니면 '정상'이라고 한 줄씩 출력하는 코드를 만들어줘.

- 90%는 연습용으로 정한 기준입니다.
- 강의 예시 코드를 실행하면 EQ-02와 EQ-04에 `점검 필요`가 붙습니다. 내 결과와 비교해 보세요.

**주의**: AI 코드가 `pass_rate = ...`처럼 같은 이름에 새 값을 넣으면 3단계 결과가 바뀝니다. 그랬다면 3단계 셀을 다시 실행하세요.

## 9단계. 연습문제 — 불량 유형별 비율(%) 계산하기
5단계의 `defect_counts`(유형별 건수)를 전체 불량 건수로 나눠 비율(%)로 바꿉니다.

- 빈칸: 전체 합계를 구하는 함수 이름(1단계에서 합격 장수를 셀 때 쓴 것)

In [ ]:
# TODO: 전체 합계를 구하는 함수 이름을 채우세요.
defect_pct = (defect_counts / defect_counts.____() * 100).round(1)
print(defect_pct)

## 10단계. AI로 재미있는 미니 프로그램 만들기
AI에게 아래처럼 요청하고, 받은 코드를 다음 셀에 붙여넣어 실행하세요.

> pass_rate라는 pandas Series에 설비별 합격률(%)이 들어 있어. 높은 순서로 정렬해서 1~3등에게 메달 이모지를 붙여 시상식처럼 발표하는 파이썬 코드를 만들어줘.

- 코드를 다 이해하지 않아도 됩니다. 처음 보는 표현은 구경만 하세요.